### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [5]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: ELEC1-Lec1-Fall 2020.pdf
  ✓ Loaded 14 pages

Processing: ELEC1-Lec2-Fall 2020.pdf
  ✓ Loaded 17 pages

Processing: ELEC1-Lec3-Fall 2020.pdf
  ✓ Loaded 22 pages

Total documents loaded: 53


In [6]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2020-10-24T14:23:53+02:00', 'title': 'ELEC 1', 'author': 'm.wahby', 'moddate': '2020-10-24T14:23:53+02:00', 'source': '..\\data\\pdfs\\ELEC1-Lec1-Fall 2020.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': 'ELEC1-Lec1-Fall 2020.pdf', 'file_type': 'pdf'}, page_content='ELECTRONICS \nFall 2020\nDr. Mona M.Soliman\nDr. Walid M. Ibrahim\nIT Dept.'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2020-10-24T14:23:53+02:00', 'title': 'ELEC 1', 'author': 'm.wahby', 'moddate': '2020-10-24T14:23:53+02:00', 'source': '..\\data\\pdfs\\ELEC1-Lec1-Fall 2020.pdf', 'total_pages': 14, 'page': 1, 'page_label': '2', 'source_file': 'ELEC1-Lec1-Fall 2020.pdf', 'file_type': 'pdf'}, page_content='2\nELECTRONICS \n\uf06e Daily life\n\uf0a8Cell phones\n\uf0a8Cars\n\uf0a8Computers\n\uf0a8TVs\n\uf0

In [7]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [8]:
chunks=split_documents(all_pdf_documents)

Split 53 documents into 53 chunks

Example chunk:
Content: ELECTRONICS 
Fall 2020
Dr. Mona M.Soliman
Dr. Walid M. Ibrahim
IT Dept....
Metadata: {'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2020-10-24T14:23:53+02:00', 'title': 'ELEC 1', 'author': 'm.wahby', 'moddate': '2020-10-24T14:23:53+02:00', 'source': '..\\data\\pdfs\\ELEC1-Lec1-Fall 2020.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': 'ELEC1-Lec1-Fall 2020.pdf', 'file_type': 'pdf'}


### Embedding And VectorStoreDB

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

e:\ROOT\work\ML_DL_Courses\agentic_ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


e:\ROOT\work\ML_DL_Courses\agentic_ai\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
e:\ROOT\work\ML_DL_Courses\agentic_ai\.venv\Lib\site-packages\huggingface_hub\file_down

Model loaded successfully. Embedding dimension: 384


C:\Users\DELL\AppData\Local\Temp\ipykernel_24556\2539499317.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [11]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [12]:
### convert the text to embeddings
texts=[doc.page_content for doc in chunks]

# Generate the embeddings
embeddings=embedding_manager.generate_embeddings(texts=texts)

# Store in the vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 53 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.44s/it]


Generated embeddings with shape: (53, 384)
Adding 53 documents to vector store...
Successfully added 53 documents to vector store
Total documents in collection: 53


### Retriever Pipeline From VectorStore

In [13]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [14]:
rag_retriever

In [16]:
rag_retriever.retrieve("tell me about electronics?")

Retrieving documents for query: 'tell me about electronics?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 81.99it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_f96ebced_0',
  'content': 'ELECTRONICS \nFall 2020\nDr. Mona M.Soliman\nDr. Walid M. Ibrahim\nIT Dept.',
  'metadata': {'source_file': 'ELEC1-Lec1-Fall 2020.pdf',
   'creationdate': '2020-10-24T14:23:53+02:00',
   'moddate': '2020-10-24T14:23:53+02:00',
   'total_pages': 14,
   'page': 0,
   'file_type': 'pdf',
   'title': 'ELEC 1',
   'producer': 'Microsoft® PowerPoint® 2013',
   'doc_index': 0,
   'author': 'm.wahby',
   'source': '..\\data\\pdfs\\ELEC1-Lec1-Fall 2020.pdf',
   'creator': 'Microsoft® PowerPoint® 2013',
   'page_label': '1',
   'content_length': 71},
  'similarity_score': 0.17805474996566772,
  'distance': 0.8219452500343323,
  'rank': 1},
 {'id': 'doc_d6e57843_31',
  'content': 'ELECTRONICS \nFall 2020\nLec. 3\nDr. Mona M.Soliman\nDr. Walid M. Ibrahim\nIT Dept.',
  'metadata': {'page_label': '1',
   'source': '..\\data\\pdfs\\ELEC1-Lec3-Fall 2020.pdf',
   'producer': 'Microsoft® PowerPoint® 2013',
   'creator': 'Microsoft® PowerPoint® 2013',
   'file_type

In [17]:
rag_retriever.retrieve("What is Electrical current?")

Retrieving documents for query: 'What is Electrical current?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.52it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_a16145d9_8',
  'content': 'Electrical Current\n9\n𝑰 𝑨 = 𝑸 𝑪𝒕 𝒔𝒆𝒄 = 𝒅𝒒𝒅𝒕\n\uf0d8 An electric current is the rate of flow of electric charge.\n\uf0d8 In electric circuits this charge is often carried \nby electrons moving through a wire. \n\uf0d8 The International System (SI) unit of electric current is \nthe ampere (A)',
  'metadata': {'doc_index': 8,
   'title': 'ELEC 1',
   'author': 'm.wahby',
   'creator': 'Microsoft® PowerPoint® 2013',
   'page_label': '9',
   'creationdate': '2020-10-24T14:23:53+02:00',
   'moddate': '2020-10-24T14:23:53+02:00',
   'source_file': 'ELEC1-Lec1-Fall 2020.pdf',
   'content_length': 270,
   'file_type': 'pdf',
   'page': 8,
   'source': '..\\data\\pdfs\\ELEC1-Lec1-Fall 2020.pdf',
   'total_pages': 14,
   'producer': 'Microsoft® PowerPoint® 2013'},
  'similarity_score': 0.47543150186538696,
  'distance': 0.524568498134613,
  'rank': 1},
 {'id': 'doc_31557b38_13',
  'content': '14\nWhat did we learn\n\uf06e Course outline \n\uf06e Electrical

### Integration VectorDB Context Pipeline with LLM Output

In [23]:
## Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm=ChatGroq(model_name="openai/gpt-oss-20b", temperature=0.1, max_tokens=1024)

def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [24]:
answer = rag_simple("What is the electrical current?", rag_retriever, llm)
answer

Retrieving documents for query: 'What is the electrical current?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 117.11it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


'The electrical current is the rate at which electric charge flows through a conductor, measured in amperes (A).'

### Enhanced RAG Pipeline Features

In [25]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is the electrical current?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is the electrical current?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 65.35it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: The electrical current is the rate at which electric charge flows through a conductor, measured in amperes (A).
Sources: [{'source': 'ELEC1-Lec1-Fall 2020.pdf', 'page': 8, 'score': 0.383892297744751, 'preview': 'Electrical Current\n9\n𝑰 𝑨 = 𝑸 𝑪𝒕 𝒔𝒆𝒄 = 𝒅𝒒𝒅𝒕\n\uf0d8 An electric current is the rate of flow of electric charge.\n\uf0d8 In electric circuits this charge is often carried \nby electrons moving through a wire. \n\uf0d8 The International System (SI) unit of electric current is \nthe ampere (A)...'}, {'source': 'ELEC1-Lec2-Fall 2020.pdf', 'page': 1, 'score': 0.18934941291809082, 'preview': 'Electrical Current\nRevision\n2\n𝑰 𝑨 = 𝑸 𝑪𝒕 𝒔𝒆𝒄 = 𝒅𝒒𝒅𝒕...'}, {'source': 'ELEC1-Lec1-Fall 2020.pdf', 'page': 13, 'score': 0.14355528354644775, 'preview': '14\nWhat did we learn\n\uf06e Course outline \n\uf06e Electrical current.\n\uf06e Electrical voltage.\n\uf06e Electrical power.\n\uf06e Two terminal ideal basic circuit element....'}]
Confidence: 0.383892297744751
Context Preview: Elec

In [26]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is the electrical current?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is the electrical current?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 61.09it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Electrical Current
9
𝑰 𝑨 = 𝑸 𝑪𝒕 𝒔𝒆𝒄 = 𝒅𝒒𝒅𝒕
 An electric current is the rate of flow of electric charge.
 In electric circuits this charge is often carried 
by electrons moving through a wire. 
 The International System (SI) unit of electric current

 is 
the ampere (A)

Electrical Current
Revision
2
𝑰 𝑨 = 𝑸 𝑪𝒕 𝒔𝒆𝒄 = 𝒅𝒒𝒅𝒕

14
What did we learn
 Course outline 
 Electrical current.
 Electrical voltage.
 Electrical power.
 Two terminal ideal basic circuit element.

Question: what is the electrical current?

Answer:

Final Answer: The electrical current is the rate of flow of electric charge—how much charge passes a point per second—measured in amperes (A).

Citations:
[1] ELEC1-Lec1-Fall 2020.pdf (page 8)
[2] ELEC1-Lec2-Fall 2020.pdf (page 1)
[3] ELEC1-Lec1-Fall 2020.pdf (page 13)
Summary: Electrical current is the flow of electric charge, defined as the amount of charge that passes a given point each second. It is measured in amperes (A).
History: {'question': 'what is the electrical current?', 'answer': 'The electrical current is the rate of flow of electric charge—how much charge passes a point per second—measured in amperes (A).', 'sources': [{'source': 'ELEC1-Lec1-Fall 2020.pdf', 'page': 8, 'score': 0.383892297744751, 'prev